# 10. Speech — sound is numbers · speech to text (Whisper) · text to speech (Supertonic)

> If a photo was a `(height, width, 3)` array of numbers, sound is **one line of numbers in time order**.
> We use models that turn those numbers into text (speech to text), and text back into those numbers (text to speech).

### What you will learn
1. Sound = sample rate (how many measurements per second) · an array of floats from −1 to 1, viewing the waveform · loudness (RMS)
2. **Speech to text (STT)**: Whisper small (INT8) — `openvino_genai.WhisperPipeline`
3. **Text to speech (TTS)**: Supertonic — **4** ONNX models run one after another with onnxruntime
4. Real-time factor (RTF): how many seconds it takes to make 1 second of audio

### Inside text to speech
```
text ─▶ [length prediction] how many seconds of speech ─▶ [text encoding] meaning of the text ─▶ [denoising × steps] carve speech shape out of noise ─▶ [vocoder] to sound numbers
         duration_predictor                                text_encoder                         vector_estimator (diffusion model)                      vocoder
```
The code is `_Supertonic` in `speech_routes.py` in the program folder — open it and read it.

## 1. Setup

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us); English notebooks live in notebooks/en/
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

import sys
import sounddevice as sd
from IPython.display import Audio
import matplotlib.pyplot as plt     # keep chart text in English

SR = 16000                          # sample rate: 16000 measurements per second (the value Whisper wants)

## 2. Record — 4 seconds
`sd.rec` only starts recording and returns right away. `sd.wait()` waits until it is done.
The result is a `(samples, channels)` array → flatten it to one line. 4 seconds × 16000 = 64000 numbers.

In [ ]:
SECONDS = 4
print("Speak now ...")
rec = sd.rec(int(SECONDS * SR), samplerate=SR, channels=1, dtype="float32")
sd.wait()
audio = rec.reshape(-1)
rms = float(np.sqrt((audio ** 2).mean()))           # loudness (average energy)
print(f"{len(audio)} samples = {len(audio) / SR:.1f} s · largest value {np.abs(audio).max():.2f} · RMS {rms:.3f}")
print("If RMS is below 0.01 it is almost silent — check the microphone")
display(Audio(audio, rate=SR))

In [ ]:
t = np.arange(len(audio)) / SR
plt.figure(figsize=(8, 2)); plt.plot(t, audio, lw=0.5); plt.xlabel("seconds"); plt.ylim(-1, 1); plt.title("waveform"); plt.show()

## 3. Speech to text — Whisper
Whisper turns the sound into a **mel spectrogram** (a time × frequency picture) and feeds it to the encoder. Then the decoder makes text tokens one by one (the same way of generating as the VLM).
If you set the language with `language="<|en|>"`, it is more accurate and faster. This lesson uses English (`lang="en"`). For Korean, use `"<|ko|>"` (`stt(audio, lang="ko")`).

In [ ]:
import openvino_genai as og

STT_DIR = MODELS / "stt"
DEVICE = "GPU" if any(d.startswith("GPU") for d in core.available_devices) else "CPU"
t0 = time.perf_counter()
whisper = og.WhisperPipeline(str(STT_DIR), DEVICE)
print(f"Loaded in {time.perf_counter() - t0:.1f} s ({DEVICE})")

def stt(a, lang="en"):                             # lang="ko" for Korean
    return str(whisper.generate(a, language=f"<|{lang}|>", task="transcribe")).strip()

t0 = time.perf_counter()
text = stt(audio)
print(f"Transcript: {text}   ({(time.perf_counter() - t0) * 1000:.0f} ms)")

## 4. Text to speech — Supertonic (4 ONNX models)
Supertonic supports many languages. This lesson uses `lang="en"`; use `lang="ko"` for Korean. The voices (`F1` … `M5`) work with any language.

In [ ]:
sys.path.insert(0, str(ROOT))                      # load speech_routes.py from the program folder
import speech_routes

tts = speech_routes._Supertonic(speech_routes.TTS_DIR)
for name, sess in [("length", tts.dp), ("encode", tts.enc), ("denoise", tts.est), ("vocoder", tts.voc)]:
    print(f"{name:6s} inputs {[i.name for i in sess.get_inputs()]}")
print("Voices:", speech_routes.VOICES, "| sample rate:", tts.sr)

In [ ]:
def say(text, voice="F1", steps=8, speed=1.05, play=True):
    t0 = time.perf_counter()
    wav = tts.say(text, lang="en", voice=voice, steps=steps, speed=speed)   # lang="ko" for Korean
    dt = time.perf_counter() - t0
    print(f"Made {len(wav) / tts.sr:.1f} s of audio in {dt:.2f} s ({len(wav) / tts.sr / dt:.1f}x real time)")
    if play:
        display(Audio(wav, rate=tts.sr))
    return wav

wav = say("Hello. This voice was made inside this computer.")

In [ ]:
# How does speed change with steps (number of denoising rounds)?
for st in (2, 4, 8, 16):
    print(f"steps={st:2d}: ", end="")
    say("Let's change the number of steps.", steps=st, play=False)

## Common errors
| Symptom | Cause · fix |
|---|---|
| `PortAudioError` | No microphone, or another program is using it. On Ubuntu, install `libportaudio2` |
| Transcript is empty | Too quiet (check RMS) · too short |
| Transcribed in the wrong language | Set `language` (`"<|en|>"`, or `"<|ko|>"` for Korean) |
| `speech_routes` import fails | Check that `EDGE_ROOT` points to the program folder (did you start with `lab.bat`?) |

## Summary
- Sound = sample rate × time floats. Loudness is measured with RMS.
- STT and VLM use the same "make tokens one by one" way of generating.
- TTS (Supertonic) is a pipeline of 4 models. `steps` trades speed for sound quality.

---
# 🛠 Mini project — Mini voice assistant

**What to build**: listen (STT) → understand the command → speak the answer (TTS). Do this 3 times.

| If you say | Answer |
|---|---|
| something with "time" in it | the current time |
| "date" · "day" | today's date |
| "plus" (e.g. 3 plus 5) | the result |
| anything else | repeat what it heard |

**What you use**: recording (`sd.rec`) · `stt` · `say` · regular expressions (`re`)

If you replace the rule-based command picking with the LLM from 08/09, it becomes a "real" assistant — see Try more.

In [ ]:
import re
from datetime import datetime

def listen(seconds=4):
    print("🎤 Listening ...")
    r = sd.rec(int(seconds * SR), samplerate=SR, channels=1, dtype="float32"); sd.wait()
    return r.reshape(-1)

def reply(heard):
    if "time" in heard or "Time" in heard:
        n = datetime.now(); return f"It is {n.hour}:{n.minute:02d} now."
    if "date" in heard or "day" in heard or "Date" in heard or "Day" in heard:
        n = datetime.now(); return f"Today is {n:%B} {n.day}."
    m = re.search(r"(\d+)\s*(?:plus|\+)\s*(\d+)", heard)   # Whisper may write "3 plus 5" or "3 + 5"
    if m:
        a, b = int(m.group(1)), int(m.group(2)); return f"{a} plus {b} is {a + b}."
    return f"You said '{heard}'." if heard else "Sorry, I didn't catch that."

for turn in range(3):
    heard = stt(listen())
    answer_text = reply(heard)
    print(f"[{turn + 1}] Heard: {heard!r} → Answer: {answer_text}")
    say(answer_text)

**Try more**
- Play the answer straight through the speaker instead of in the notebook: `sd.play(wav, tts.sr); sd.wait()`
- Make it stop when you say "stop"
- Let an LLM make the answer instead of rules (`pipe.generate(question)` from lesson 09) — but it uses a lot of memory